# LFM2.5-2.6B tool use with `llama-cpp-python`

This notebook demonstrates Liquid AI's **LFM2.5-2.6B** model in Google Colab. It loads the Q4_K_M GGUF checkpoint with `llama-cpp-python`, verifies ordinary chat generation, and then runs a multi-tool example.

> **GPU required:** In Colab, select **Runtime > Change runtime type > T4 GPU** (or a newer GPU) before running the notebook. A CPU-only runtime is not supported by this walkthrough.

The tool-calling example follows Liquid AI's [tool-use workflow](https://docs.liquid.ai/lfm/key-concepts/tool-use): describe the available functions, let the model request them, execute them in Python, return their results as tool messages, and ask the model to produce a final answer.

## 1. Inspect the Colab GPU

Run `nvidia-smi` to confirm that the Colab runtime has an NVIDIA GPU attached and to inspect its driver, CUDA version, and available memory. The GPU should be an NVIDIA **T4 or better**; if no GPU appears, change the Colab runtime type before continuing.

In [1]:
!nvidia-smi


Sun Aug 16 08:22:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install the CUDA-enabled `llama-cpp-python` wheel

Remove any existing CPU build and install the pinned CUDA 12.1 wheel directly. Using the wheel's URL prevents pip from selecting the CPU package from PyPI. CUDA 12.1 binaries are compatible with the newer NVIDIA driver provided by the Colab T4 runtime.

> **Important:** If `llama_cpp` has already been imported in this session, run this installation cell and then select **Runtime > Restart session** before continuing. Native libraries already loaded by Python cannot be replaced inside the running process.

In [33]:
!pip uninstall -qqq -y llama-cpp-python
!pip install -qqq --no-cache-dir https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.34-cu121/llama_cpp_python-0.3.34-py3-none-manylinux_2_35_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 237.0 MB/s eta 0:00:0000:0100:01


## 3. Load LFM2.5-2.6B on the GPU

Download the **Q4_K_M** GGUF checkpoint from `LiquidAI/LFM2.5-2.6B-GGUF` and initialize a 16,384-token context. `n_gpu_layers=-1` requests full GPU offload. The load log should report that all model layers were offloaded to CUDA, and the final `nvidia-smi` output should show VRAM used by the Python process.

In [2]:
import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

from llama_cpp import Llama

llm = Llama.from_pretrained(
  repo_id="LiquidAI/LFM2.5-2.6B-GGUF",
  filename="LFM2.5-2.6B-Q4_K_M.gguf",
  n_gpu_layers=-1,
  n_ctx=16384,
)

print("\nGPU status after loading the model:")
!nvidia-smi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
llama_model_loader: loaded meta data with 33 key-value pairs and 266 tensors from /root/.cache/huggingface/hub/models--LiquidAI--LFM2.5-2.6B-GGUF/snapshots/b421ad1d549afeda6a0fb2ad3a697cb5a7879adc/./LFM2.5-2.6B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = lfm2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sam


GPU status after loading the model:
Sun Aug 16 08:23:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             29W /   70W |    3029MiB /  15360MiB |      1%      Default |
|                                         |                        |                  N/A |
+----------

## 4. Verify basic chat generation

Send a simple question through the checkpoint's built-in chat template. The low-temperature sampling settings make the response more focused and reproducible, and `max_tokens=2048` sets a generous upper bound for the answer within the 16,384-token context. This cell returns the complete `llama-cpp-python` response object; check `finish_reason` to distinguish a natural stop from a response that reached the token limit.

In [3]:
llm.create_chat_completion(
  messages = [{"role": "user", "content": "What is AI"}],
  temperature=0.1,
  top_k=50,
  top_p=0.1,
  repeat_penalty=1.05,
  max_tokens=2048,
)

ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =     315.00 ms /    13 tokens (   24.23 ms per token,    41.27 tokens per second)
llama_perf_context_print:        eval time =   11830.58 ms /  1258 runs   (    9.40 ms per token,   106.33 tokens per second)
llama_perf_context_print:       total time =   16448.28 ms /  1271 tokens
llama_perf_context_print:    graphs reused =       1253


{'id': 'chatcmpl-11cb2eb2-1872-488c-b4d0-dd13997af291',
 'object': 'chat.completion',
 'created': 1786868595,
 'model': '/root/.cache/huggingface/hub/models--LiquidAI--LFM2.5-2.6B-GGUF/snapshots/b421ad1d549afeda6a0fb2ad3a697cb5a7879adc/./LFM2.5-2.6B-Q4_K_M.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'The user is asking "What is AI". This is a broad, fundamental question. I need to provide a clear, accurate, and comprehensive definition.\n\n**Key concepts to cover:**\n1.  **Definition:** Artificial Intelligence (AI) stands for Artificial Intelligence. It\'s the simulation of human intelligence in machines.\n2.  **Core Idea:** Machines are made to think, learn, and solve problems like humans do.\n3.  **Types/Levels:**\n    *   Narrow AI (Weak AI): Designed for specific tasks (e.g., chess bots, Siri). This is what most people mean when they say "AI" today.\n    *   General AI (Strong AI): Hypothetical AI with human-like cognitive abilities across al

## 5. Benchmark inference performance

Warm up the model, then time three identical chat completions. The benchmark reports end-to-end latency and completion tokens per second for each run, followed by averages. Because elapsed time includes both prompt processing and token generation, the reported throughput represents the complete request rather than decode-only speed.

The 128-token output cap keeps the benchmark reasonably quick. A `finish_reason` of `length` is expected when the model fills that cap; adjust `benchmark_runs` and `benchmark_max_tokens` to test a different workload.

In [4]:
import statistics
import time

benchmark_runs = 3
benchmark_max_tokens = 128
benchmark_messages = [
    {"role": "user", "content": "Explain how neural networks learn, using complete sentences."}
]

# Warm up model execution and GPU kernels before measuring.
print("Warming up...")
llm.create_chat_completion(
    messages=benchmark_messages,
    temperature=0.0,
    max_tokens=16,
)

measurements = []
for run_number in range(1, benchmark_runs + 1):
    started_at = time.perf_counter()
    benchmark_response = llm.create_chat_completion(
        messages=benchmark_messages,
        temperature=0.0,
        max_tokens=benchmark_max_tokens,
    )
    elapsed_seconds = time.perf_counter() - started_at

    usage = benchmark_response.get("usage", {})
    completion_tokens = usage.get("completion_tokens", 0)
    tokens_per_second = completion_tokens / elapsed_seconds if elapsed_seconds else 0.0
    finish_reason = benchmark_response["choices"][0].get("finish_reason")
    measurements.append((elapsed_seconds, tokens_per_second))

    print(
        f"Run {run_number}: {elapsed_seconds:.2f} s | "
        f"{completion_tokens} completion tokens | "
        f"{tokens_per_second:.2f} tokens/s | finish_reason={finish_reason}"
    )

average_latency = statistics.mean(item[0] for item in measurements)
average_throughput = statistics.mean(item[1] for item in measurements)
print(f"\nAverage latency: {average_latency:.2f} s")
print(f"Average end-to-end throughput: {average_throughput:.2f} tokens/s")

Llama.generate: 4 prefix-match found but partial kv removal not supported, re-evaluating full prompt
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture


Warming up...


llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =     191.84 ms /    20 tokens (    9.59 ms per token,   104.26 tokens per second)
llama_perf_context_print:        eval time =     188.69 ms /    15 runs   (   12.58 ms per token,    79.50 tokens per second)
llama_perf_context_print:       total time =     389.60 ms /    35 tokens
llama_perf_context_print:    graphs reused =         14
Llama.generate: 19 prefix-match found but partial kv removal not supported, re-evaluating full prompt
llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =      24.29 ms /    20 tokens (    1.21 ms per token,   823.38 tokens per second)
llama_perf_context_print:        eval time =    1359.91 ms /   127 runs   (   10.71 ms per token,    93.39 tokens per second)
llama_perf_context_print:       total time =    1450.94 ms /   147 tokens
llama_perf_context_print:    graphs reused =        126
Llama.generate: 

Run 1: 1.46 s | 128 completion tokens | 87.97 tokens/s | finish_reason=length


llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =      24.04 ms /    20 tokens (    1.20 ms per token,   831.88 tokens per second)
llama_perf_context_print:        eval time =    1379.87 ms /   127 runs   (   10.87 ms per token,    92.04 tokens per second)
llama_perf_context_print:       total time =    1467.34 ms /   147 tokens
llama_perf_context_print:    graphs reused =        126
Llama.generate: 19 prefix-match found but partial kv removal not supported, re-evaluating full prompt


Run 2: 1.47 s | 128 completion tokens | 86.82 tokens/s | finish_reason=length


llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =      23.31 ms /    20 tokens (    1.17 ms per token,   858.00 tokens per second)
llama_perf_context_print:        eval time =    1390.29 ms /   127 runs   (   10.95 ms per token,    91.35 tokens per second)
llama_perf_context_print:       total time =    1478.72 ms /   147 tokens
llama_perf_context_print:    graphs reused =        126


Run 3: 1.48 s | 128 completion tokens | 86.33 tokens/s | finish_reason=length

Average latency: 1.47 s
Average end-to-end throughput: 87.04 tokens/s


## 6. Run a multi-tool request

Define two tools—`get_weather` and `get_local_time`—and include their JSON schemas in the system prompt. The user asks the model to call both tools before recommending what to do in Singapore.

The cell performs the full tool-use loop:

1. Generate the model's tool requests.
2. Read structured `tool_calls`, or safely recover Python-style calls from message text when the GGUF chat handler returns them there.
3. Execute local demo implementations: a fixed weather result and the current Singapore time from Python's timezone database.
4. Append the tool results to the conversation and generate a concise final answer.

> **Note:** The weather function is a deterministic mock for demonstration purposes. Replace it with a real weather service in a production application.

In [5]:
import ast
import json
import re

tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"}
            },
            "required": ["city"],
        },
    },
    {
        "name": "get_local_time",
        "description": "Get the current local time for an IANA timezone.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {"type": "string", "description": "IANA timezone, such as Asia/Singapore"}
            },
            "required": ["timezone"],
        },
    }
]

messages = [
    {
        "role": "system",
        "content": (
            f"List of tools: {json.dumps(tools)}\n"
            "Use both tools before answering. Output function calls in the Pythonic format described by the tool definitions."
        ),
    },
    {
        "role": "user",
        "content": "I am planning a trip to Singapore. Use BOTH get_weather and get_local_time, then give me a concise recommendation.",
    },
]

try:
    response = llm.create_chat_completion(
        messages=messages,
        temperature=0.1,
        top_k=50,
        top_p=0.1,
        repeat_penalty=1.05,
        max_tokens=1024,
    )
    message = response["choices"][0]["message"]
    tool_calls = message.get("tool_calls", [])

    # Some GGUF chat templates emit calls as text instead of structured tool_calls.
    if not tool_calls and message.get("content"):
        text_calls = re.findall(
            r"\b(get_weather|get_local_time)\(([^()]*)\)",
            message["content"],
        )
        for index, (name, argument_text) in enumerate(text_calls):
            try:
                expression = ast.parse(f"f({argument_text})", mode="eval").body
                arguments = {
                    keyword.arg: ast.literal_eval(keyword.value)
                    for keyword in expression.keywords
                }
                tool_calls.append({
                    "id": f"text_call_{index}",
                    "type": "function",
                    "function": {
                        "name": name,
                        "arguments": arguments,
                    },
                })
            except (SyntaxError, ValueError):
                pass

    if not tool_calls:
        print("No tool call was generated.")
        print("Assistant output:", message.get("content"))
    else:
        print(f"Generated {len(tool_calls)} tool call(s):")
        # Normalize the assistant message so the follow-up works for both formats.
        assistant_message = dict(message)
        assistant_message["content"] = None
        assistant_message["tool_calls"] = tool_calls
        messages.append(assistant_message)

        for index, call in enumerate(tool_calls):
            function = call["function"]
            name = function["name"]
            raw_arguments = function["arguments"]
            arguments = raw_arguments if isinstance(raw_arguments, dict) else json.loads(raw_arguments)
            print(f"- {name}({arguments})")

            # Local demo implementations; replace these with real APIs in production.
            if name == "get_weather":
                result = {"city": arguments["city"], "condition": "partly cloudy", "temperature_c": 31}
            elif name == "get_local_time":
                from datetime import datetime
                from zoneinfo import ZoneInfo
                result = {"timezone": arguments["timezone"], "local_time": datetime.now(ZoneInfo(arguments["timezone"])).isoformat()}
            else:
                result = {"error": f"Unknown tool: {name}"}

            messages.append({
                "role": "tool",
                "tool_call_id": call.get("id", f"call_{index}"),
                "content": json.dumps(result),
            })

        final_response = llm.create_chat_completion(
            messages=messages,
            temperature=0.0,
            max_tokens=2048,
        )
        print("\nFinal answer:")
        print(final_response["choices"][0]["message"].get("content"))
except Exception as exc:
    print(f"Tool-call test failed: {type(exc).__name__}: {exc}")
    print("This usually means the model chat template does not support tool calling.")

Llama.generate: 2 prefix-match found but partial kv removal not supported, re-evaluating full prompt
llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =     123.99 ms /   187 tokens (    0.66 ms per token,  1508.23 tokens per second)
llama_perf_context_print:        eval time =    1202.28 ms /   123 runs   (    9.77 ms per token,   102.31 tokens per second)
llama_perf_context_print:       total time =    1617.04 ms /   310 tokens
llama_perf_context_print:    graphs reused =        121
Llama.generate: 186 prefix-match found but partial kv removal not supported, re-evaluating full prompt


Generated 2 tool call(s):
- get_weather({'city': 'Singapore'})
- get_local_time({'timezone': 'Asia/Singapore'})


llama_perf_context_print:        load time =     317.00 ms
llama_perf_context_print: prompt eval time =     106.73 ms /   283 tokens (    0.38 ms per token,  2651.67 tokens per second)
llama_perf_context_print:        eval time =    2770.93 ms /   253 runs   (   10.95 ms per token,    91.31 tokens per second)
llama_perf_context_print:       total time =    3028.87 ms /   536 tokens
llama_perf_context_print:    graphs reused =        251



Final answer:
The user wants a concise recommendation for their trip to Singapore. I have the weather and local time information:

- Weather: Partly cloudy, 31°C
- Local time: 2026-08-16T16:23:59.317991+08:00 (which is 4:23 PM Singapore time)

Based on this, I can provide a recommendation. The temperature is quite warm (31°C), so I should suggest staying hydrated and perhaps bringing light clothing. The partly cloudy condition is pleasant. Since it's mid-afternoon, they might want to plan outdoor activities accordingly.

Let me craft a concise recommendation that incorporates both pieces of information.</think>**Recommendation for your Singapore trip:**

- **Weather:** It's currently **partly cloudy and 31°C** — warm and humid, typical for Singapore in August.
- **Time:** It's **4:23 PM local time** (Asia/Singapore).

**Suggestion:** Pack lightweight, breathable clothing and stay hydrated. The partly cloudy skies make for pleasant outdoor sightseeing, but be prepared for the heat. If 